# TTS bootcamp on Kaggle: Task 1 scoring + Task 2 audio

Before running: **Settings → Accelerator = GPU T4**, **Internet = On** (needs phone-verified account),
and **Add-ons → Secrets → HF_TOKEN** (your Hugging Face token, with the IndicF5 terms accepted).
Attach the dataset made from `tts_kit.zip` (Add Input). Then **Run All**. Every heavy step runs in its own
subprocess, so no kernel restart is needed.

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path

src = next(p.parent for p in Path("/kaggle/input").rglob("metrics.py"))
KIT = Path("/kaggle/working/tts_kit")
if not KIT.exists():
    shutil.copytree(src, KIT)
os.chdir(KIT)
print("kit:", KIT, "->", sorted(os.listdir(KIT)))

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("Could not read the HF_TOKEN secret (" + type(e).__name__ + "). Add it under Add-ons > Secrets and tick it,")
    print("or paste the token below for this session only (it is not saved anywhere).")
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ").strip()
assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN missing or malformed"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
Path("out").mkdir(exist_ok=True)

def sh(cmd, retry_on=None, show=12000):
    """Run a command in a fresh process; if it asks for the one-time length-fix restart, simply run it again."""
    print("$", cmd[:200])
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode and retry_on and retry_on in out:
        print("(length fix was just applied: running again, this happens once)")
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        out = (r.stdout or "") + (r.stderr or "")
    print(out[-show:])
    if r.returncode:
        raise RuntimeError(f"command failed (exit {r.returncode}): {cmd[:120]}")
    return out

In [ ]:
print(sh("nvidia-smi -L"))
print(sh("python -c \"import torch,sys;print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())\""))

## 1. Install (numpy pinned as the kit says; IndicF5 from GitHub main with --no-deps because its setup.py wants numpy<=1.26)

In [ ]:
sh("pip install -q 'numpy>=2.1,<2.3' 'scipy>=1.15,<2' 'transformers==4.49.0' accelerate soundfile librosa pyyaml pandas 'speechbrain>=1.0' huggingface_hub")
sh("pip install -q --no-deps git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7")
sh("pip install -q cached_path click datasets ema_pytorch hydra-core jieba matplotlib pydub pypinyin safetensors tomli torchdiffeq tqdm transformers_stream_generator vocos x_transformers")
sh("python -c \"import numpy,transformers;print('numpy',numpy.__version__,'transformers',transformers.__version__)\"")

## 2. Task 1: score the provided baseline audio (reproduce CER 0.2421 / SIM 0.8303)

In [ ]:
out = sh("python metrics.py data/baseline_val --json out/baseline_val.json --csv out/baseline_val_scores.csv")
Path("out/metrics_baseline.log").write_text(out, encoding="utf-8")

## 3. Task 2: synthesize the 40 evaluation items from the normalized text

In [ ]:
sh("python baseline/run_baseline.py --items data/eval_items.csv --out predictions/t2_norm "
   "--text-csv predictions/t2_normalized.csv --text-column normalized_text --seed 0", retry_on="length fix")
print(len(list(Path("predictions/t2_norm").glob("*.wav"))), "wav files in predictions/t2_norm")

## 4. Task 2 on validation: normalized text, everything else baseline (this is also `val_control/` for Task 3)

In [ ]:
sh("python baseline/run_baseline.py --items data/val_items.csv --out predictions/val_control "
   "--text-csv predictions/t2_normalized.csv --text-column normalized_text --seed 0", retry_on="length fix")
out = sh("python metrics.py predictions/val_control --json out/val_control.json --csv out/val_control_scores.csv")
Path("out/metrics_val_control.log").write_text(out, encoding="utf-8")

## 5. Bundle everything to download (Output panel on the right, or after Save Version)

In [ ]:
import zipfile
zp = Path("/kaggle/working/tts_outputs.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["predictions/t2_norm/*.wav", "predictions/val_control/*.wav", "out/*", "out/*.json"]:
        for p in Path(".").glob(pat):
            if p.is_file():
                z.write(p, p.as_posix())
print(zp, round(zp.stat().st_size / 1e6, 1), "MB")
print(json.dumps(json.load(open("out/baseline_val.json")), ensure_ascii=False, indent=1))